In [122]:
import pandas as pd
import numpy as np
import os

In [ ]:
data = pd.read_csv("..\\Dataset\\NYC_DWT_Analysis\\NYC_Drinking_Water_Tank_Inspections_and_Audits_Compliance_Results_20251203.csv")
cols = data.columns
print(cols)

Index(['BIN', 'House #', 'Street Name', 'Zip_code', 'Borough', 'Status',
       'Number of DWT', 'Activity_Type', 'Activity_Year', 'Violation_Code',
       'Law_Section', 'Violation_Text', 'Compliance_Year',
       'Date_of_Occurrence', 'Summons_Number', 'BBL', 'Longitude', 'Latitude',
       'Community_Board', 'Council_District', 'Census_Tract', 'NTA_Code'],
      dtype='object')


In [118]:
#Verifying accurate lat/long->borough mapping is not possible, infromation is not present online.len
# Closest source: https://codelibrary.amlegal.com/codes/newyorkcity/latest/NYCadmin/0-0-0-95
# Uses relative coords and landmarks, unusable (also over 100 years old) 

#The highNaN columns indicate violation information, indicating that their NaNs are not missing data but rather passed inspections
#Populate NaNs with 0
highNaNCols = data.columns[data.isna().mean() > 0.3].tolist()
print(highNaNCols)

for c in highNaNCols:
    data[c].fillna(0, inplace=True)

['Violation_Code', 'Law_Section', 'Violation_Text', 'Date_of_Occurrence', 'Summons_Number']


C:\Users\adity\AppData\Local\Temp\ipykernel_2472\4059763772.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[c].fillna(0, inplace=True)


In [119]:
#Forcing all strings to lowercase
for c in data.columns:
    if data[c].dtype == "object":
        data[c] = data[c].astype(str).str.lower()

In [120]:
#Dropping rows with NaNs
nan_rows = data[data.isna().any(axis=1)]
null_rows = data[data.isnull().any(axis=1)]
error_rows = data[data["Borough"] == "error: #n/a"]
rowsToDrop = pd.concat([nan_rows, null_rows, error_rows]).drop_duplicates()

print(f"Dropping {len(rowsToDrop)} rows due to null/error values.")

data = data.drop(rowsToDrop.index, axis=0)


Dropping 11 rows due to null/error values.


In [ ]:
#Splitting data per-year and per-borough
years = data["Activity_Year"].unique()
boroughs = data["Borough"].unique()

print("Years for which data is present:")
year_data = {}
for y in years:
    year_data[y] = len(data[data["Activity_Year"] == y])
    print(f"Inspections in {y}: {year_data[y]}")


#2024 Data will be ignored going forward, due to ridiculously low number of inspections
#So will Staten Island data, single digit number of inspections
borough_dfs = {}
for y in years:
    print('---------------------------------------')
    os.mkdir(f"..\\Dataset\\NYC_DWT_Analysis\\{y}")
    for b in boroughs:
        borough_dfs[(b, y)] = data[(data["Borough"] == b) & (data["Activity_Year"] == y)]
        print(f"Inspections in {b} in {y}: {len(borough_dfs[(b, y)])}")
        if (borough_dfs[(b, y)].to_csv(f"..\\Dataset\\NYC_DWT_Analysis\\{y}\\{b}.csv")== None): print(f"{b}_{y} written to .csv\n")

Years for which data is present:
Inspections in 2021.0: 5622
Inspections in 2024.0: 30
Inspections in 2022.0: 5508
Inspections in 2023.0: 5391
---------------------------------------
Inspections in manhattan in 2021.0: 4742
manhattan_2021.0 written to .csv

Inspections in brooklyn in 2021.0: 353
brooklyn_2021.0 written to .csv

Inspections in bronx in 2021.0: 283
bronx_2021.0 written to .csv

Inspections in queens in 2021.0: 236
queens_2021.0 written to .csv

Inspections in staten island in 2021.0: 8
staten island_2021.0 written to .csv

---------------------------------------
Inspections in manhattan in 2024.0: 15
manhattan_2024.0 written to .csv

Inspections in brooklyn in 2024.0: 5
brooklyn_2024.0 written to .csv

Inspections in bronx in 2024.0: 10
bronx_2024.0 written to .csv

Inspections in queens in 2024.0: 0
queens_2024.0 written to .csv

Inspections in staten island in 2024.0: 0
staten island_2024.0 written to .csv

---------------------------------------
Inspections in manhatt